# Notebook For Analysis

# Setup

In [1]:
!echo $HOSTNAME
!python --version
!nvidia-smi

g006
Python 3.12.5
Wed Jul  1 12:08:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:CA:00.0 Off |                    0 |
| N/A   25C    P0             34W /  250W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

In [2]:
# General Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import os
import sys
import tqdm.auto as tqdm
import random
from pathlib import Path
import plotly.express as px
import copy

from typing import List, Union, Optional
from functools import partial
import itertools
from IPython.display import HTML

# CSV Use Libraries
import pandas as pd

# Function Imports
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Unused
# from torch.utils.data import DataLoader
# from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
# import dataclasses
# import datasets
# import ast
# from transformers import get_cosine_schedule_with_warmup #Used for lr?
# from torch.nn.utils import clip_grad_norm_

In [3]:
from config import *

# Shared helpers live in Transformer.py (the training script). Importing them does
# NOT trigger training — that is guarded by `if __name__ == "__main__"`.
from Transformer import (
    setup_device, build_model, load_and_split_data, load_checkpoint_into,
    descent_loss_fn, descent_accuracy_fn, descent_sequence_accuracy,
    create_attention_mask, register_pad_mask_hook,
)

## Define Graphing Functs

In [4]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")

Using renderer: notebook_connected


In [5]:
pio.templates['plotly'].layout.xaxis.title.font.size = 20
pio.templates['plotly'].layout.yaxis.title.font.size = 20
pio.templates['plotly'].layout.title.font.size = 30

In [ ]:
import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens import HookedTransformer

# Unpickling shim: checkpoints pickled by other transformer_lens versions may reference
# the config class through the old top-level module path. Alias it if this version
# provides the new path; skip silently if the layout changes again.
try:
    import transformer_lens.config.hooked_transformer_config as htc
    sys.modules.setdefault('transformer_lens.HookedTransformerConfig', htc)
except ImportError:
    pass

In [7]:
# Unused plotting functions in favor of Neel Plotly

# def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
#     px.imshow(utils.to_numpy(tensor), color_continuous_midpoint=0.0, color_continuous_scale="RdBu", labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

# def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
#     px.line(utils.to_numpy(tensor), labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

# def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
#     x = utils.to_numpy(x)
#     y = utils.to_numpy(y)
#     px.scatter(y=y, x=x, labels={"x":xaxis, "y":yaxis, "color":caxis}, **kwargs).show(renderer)

## Set Paths and GPU Devices

In [8]:
# PTH_LOCATION is imported from config.py; create the directory if it doesn't exist yet
# DATA_PATH and PTH_LOCATION are now resolved from config.py
os.makedirs(Path(PTH_LOCATION).parent, exist_ok=True)
print(f"Data Dir:  {DATA_PATH}")
print(f"Model PTH: {PTH_LOCATION}")

Data Dir:  /home/linrya/Runs/CayleyGraph/Transformer
Model PTH: /home/linrya/Runs/CayleyGraph/Transformer/workspace/_scratch/model.pth


In [9]:
device, device1 = setup_device()

CUDA available: True
Device count: 1
Device 0: NVIDIA A100-PCIE-40GB
Device Name: cuda


## Define Model and Optimizer Params

In [ ]:
torch.manual_seed(seed=DATA_SEED)
torch.cuda.manual_seed_all(DATA_SEED)

# Load architecture + weights from the saved checkpoint. build_model() reconstructs
# the model/optimizer/scheduler (and disables biases); load_checkpoint_into() fills in
# the trained state and returns the training-curve history.
cached_data = torch.load(PTH_LOCATION, weights_only=False)
cfg = cached_data["config"]

model, optimizer, scheduler = build_model(cfg)
_, history = load_checkpoint_into(model, optimizer, scheduler)

model_checkpoints = history["checkpoints"]
checkpoint_epochs = history["checkpoint_epochs"]
train_losses      = history["train_losses"]
test_losses       = history["test_losses"]
train_accuracies  = history["train_accuracies"]
test_accuracies   = history["test_accuracies"]
train_bit_accuracies = history["train_bit_accuracies"]
test_bit_accuracies  = history["test_bit_accuracies"]

## Misc Setup
Loss/accuracy/masking functions are imported from `Transformer.py` at the top.

In [13]:
torch.cuda.empty_cache()
clsDex = 21 # index of classification token, kept for attention pattern analysis

## Initialize Datasets

In [14]:
# Load + shuffle + split + move-to-device, using the same seeded split as training.
data = load_and_split_data(device1)
train_tokens,  test_tokens  = data["train_tokens"],  data["test_tokens"]
train_targets, test_targets = data["train_targets"], data["test_targets"]
train_mask,    test_mask    = data["train_mask"],    data["test_mask"]
train_attention_mask = data["train_attention_mask"]
test_attention_mask  = data["test_attention_mask"]

FileNotFoundError: [Errno 2] No such file or directory: '/home/linrya/Runs/CayleyGraph/Transformer/data.csv'

# Graph Results

In [ ]:
from neel_plotly.plot import line_or_scatter, line as neel_line

In [ ]:
print(CHECKPOINT_STEP)
skipBy = 1


def createGraph(yTrain, yTest, yName, title):
    fig = line_or_scatter(
        [yTrain[::skipBy], yTest[::skipBy]],
        x=np.arange(0, len(yTrain), skipBy),
        xaxis="Epoch",
        yaxis=yName,
        log_y=True,
        title=title,
        line_labels=['train', 'test'],
        toggle_x=True,
        toggle_y=True,
        plot_type="line",
        return_fig=True
    )
    return fig

fig1 = createGraph(train_losses, test_losses, "Loss", "Loss Curve for Word Problem")
fig2 = createGraph(train_accuracies, test_accuracies, "Accuracy", "Accuracy Curve for Word Problem")
fig3 = createGraph(train_bit_accuracies, test_bit_accuracies, "Per Bit Accuracy", "Per Bit Accuracy Curve for Word Problem")


fig1.show()
fig2.show()
fig3.show()


fig1.write_html("1. loss_curve.html")
fig2.write_html("2. accuracy_curve.html")
fig3.write_html("3. bit_accuracy.html")


# Analysing the Model

Helpful Memory Probing Functions:

- nvidia-smi
- torch.cuda.empty_cache()
- torch.set_grad_enabled(mode=False)
- gc.collect()
- print(torch.cuda.memory_summary())
- torch profiler also (saved image on Aug 03, 2025 ~6pm)

### Helper Functions
- logit return function
- prediction function

In [ ]:
import gc

def getLogits(model: HookedTransformer, tokens, getCache: bool = False):
    """
    Applies the padding mask and runs a forward pass.
    Returns detached logits (and optionally the activation cache).
    """
    with torch.inference_mode():
        model.reset_hooks()
        mask = create_attention_mask(tokens).to(device1)
        register_pad_mask_hook(model, mask)
        if getCache:
            original_logits_og, cache = model.run_with_cache(tokens)
        else:
            original_logits_og = model(tokens)
    original_logits = original_logits_og.detach().clone()
    del original_logits_og, mask
    torch.cuda.empty_cache()
    if getCache:
        return original_logits, cache
    else:
        return original_logits

def getPredictions(model: HookedTransformer, tokens, targets, mask):
    """
    Per-sequence descent accuracy: fraction of prefixes whose descent set is
    exactly correct. Shape: (batch_size,)
    """
    logits  = getLogits(model, tokens, getCache=False)
    seq_acc = descent_sequence_accuracy(logits, targets, mask)
    del logits
    return seq_acc

def imshow(tensor, renderer=None, xaxis="", yaxis="", xlabels=None, ylabels=None, aspect="auto", **kwargs):
    fig = px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        aspect=aspect,
        **kwargs
    )
    if xlabels is not None:
        fig.update_xaxes(tickmode='array', tickvals=list(range(len(xlabels))), ticktext=xlabels)
    if ylabels is not None:
        fig.update_yaxes(tickmode='array', tickvals=list(range(len(ylabels))), ticktext=ylabels)
    fig.update_yaxes(scaleanchor=None)
    fig.show(renderer)
    return fig

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

## Quick Analysis + Memory Cleanup

In [ ]:
torch.set_grad_enabled(False)
gc.collect()
torch.cuda.empty_cache()

In [ ]:
original_logits, cache = getLogits(model, train_tokens, getCache=True)

Get key weight matrices:

In [ ]:
W_E = model.embed.W_E[:-1]
print("W_E", W_E.shape)
W_neur = W_E @ model.blocks[0].attn.W_V @ model.blocks[0].attn.W_O @ model.blocks[0].mlp.W_in
print("W_neur", W_neur.shape)
W_logit = model.blocks[0].mlp.W_out @ model.unembed.W_U
print("W_logit", W_logit.shape)

In [ ]:
original_loss = descent_loss_fn(original_logits, train_targets, train_mask).item()
print("Original Loss:", original_loss)

### Looking at Activations

Get all shapes:

In [ ]:
for param_name, param in cache.items():
    print(param_name, param.shape)

In [ ]:
print(model)

# debug: prints the first 4 input train data values
print(train_tokens[:4])

# Embedding matrix W_E: [d_vocab, d_model] = (4, 256) — 3 generators + the padding token.
# The last row is the padding token's embedding; [:-1] drops it.
print(model.embed.W_E.shape)
print(model.embed.W_E)
print(model.embed.W_E[:-1])

# Embedding a batch of tokens: [batch, seq_len] -> [batch, seq_len, d_model]
print(train_tokens.shape)
print(model.embed(train_tokens).shape)

## Attention Heads (average)

### Average attention over all words for each head
Note: all train data input words used and averaged out for these graphs
- x: letters (keys attended to)
- y: letters (query giving attention to x axis)

In [26]:
n_heads = model.cfg.n_heads
seq_len = model.cfg.n_ctx
str_tokens = [str(i) for i in range(seq_len)]

with torch.inference_mode():
    for layer in range(model.cfg.n_layers):
        head_sums  = None   # reset per layer
        num_samples = 0

        for wordIndex in range(len(train_tokens)):
            # Grab attention pattern: [n_heads, seq_len, seq_len]
            attn_patterns = cache["pattern", layer][wordIndex]

            if head_sums is None:
                head_sums = torch.zeros_like(attn_patterns)

            head_sums   += attn_patterns
            num_samples += 1

        # Average attention
        avg_attn = head_sums / num_samples

        for head in range(n_heads):
            imshow(
                avg_attn[head].detach().cpu(),
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} — Average Attention Across Dataset",
                aspect=None,
            )

### Average Attention Pattern over Attention Heads

In [34]:
for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer].mean(dim=0)[:, clsDex, :]
    imshow(
        attention,
        title=f"Average Attention Paid | for token {clsDex} | per head | layer {layer}",
        xaxis="Source",
        yaxis="Head",
        x=[str(i) for i in range(train_tokens.shape[1])],
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

## Attention Heads (word: X)

In [59]:
# Look at a specific word in the dataset by its (shuffled) index
wordIndex = 100

# Evaluate the model on a single word
wordTensor  = train_tokens[wordIndex]
wordTokens  = wordTensor.unsqueeze(0)                  # add batch dimension
wordTargets = train_targets[wordIndex].unsqueeze(0)
wordMask    = train_mask[wordIndex].unsqueeze(0)
seq_acc = getPredictions(model, wordTokens, wordTargets, wordMask)

logits = getLogits(model, wordTokens, getCache=False)
probs = torch.sigmoid(logits)
preds = (probs > 0.5).int()

print(f"Word:    {wordTensor.tolist()}")

for t in range(wordTokens.shape[1]):
    true_labels = wordTargets[0, t].int().tolist()
    pred_labels = preds[0, t].tolist()

    print(f"{t:02d}  True: {true_labels}   Pred: {pred_labels}   Correct:{true_labels == pred_labels}")

print(f"Seq Acc: {seq_acc.item():.4f}  (fraction of prefixes with an exactly-correct descent set)")

Word:    [1, 1, 2, 2, 1, 1, 3, 2, 3, 3, 3, 2, 3, 1, 2, 3, 1, 1, 2, 1, 1, 2]
00  True: [1, 0, 0]   Pred: [1, 0, 0]   Correct:True
01  True: [0, 0, 0]   Pred: [0, 0, 0]   Correct:True
02  True: [0, 1, 0]   Pred: [0, 1, 0]   Correct:True
03  True: [0, 0, 0]   Pred: [0, 0, 0]   Correct:True
04  True: [1, 0, 0]   Pred: [1, 0, 0]   Correct:True
05  True: [0, 0, 0]   Pred: [0, 0, 0]   Correct:True
06  True: [0, 0, 1]   Pred: [0, 0, 1]   Correct:True
07  True: [0, 1, 0]   Pred: [0, 1, 0]   Correct:True
08  True: [0, 1, 1]   Pred: [0, 1, 1]   Correct:True
09  True: [0, 1, 0]   Pred: [0, 1, 0]   Correct:True
10  True: [0, 1, 1]   Pred: [0, 1, 1]   Correct:True
11  True: [0, 0, 1]   Pred: [0, 0, 1]   Correct:True
12  True: [0, 1, 0]   Pred: [0, 1, 1]   Correct:False
13  True: [1, 0, 0]   Pred: [1, 0, 0]   Correct:True
14  True: [1, 1, 0]   Pred: [1, 1, 0]   Correct:True
15  True: [0, 0, 1]   Pred: [0, 0, 1]   Correct:True
16  True: [1, 0, 1]   Pred: [1, 0, 0]   Correct:False
17  True: [0, 0, 1]  

In [ ]:
# Look at a specific word in the dataset by its (shuffled) index
testIndex = 0

# Evaluate the model on a single word
wordTensor  = test_tokens[testIndex]
wordTokens  = wordTensor.unsqueeze(0)                  # add batch dimension
wordTargets = test_targets[testIndex].unsqueeze(0)
wordMask    = test_mask[testIndex].unsqueeze(0)
seq_acc = getPredictions(model, wordTokens, wordTargets, wordMask)

logits = getLogits(model, wordTokens, getCache=False)
probs = torch.sigmoid(logits)
preds = (probs > 0.5).int()

print("Word:    ", end="")
for i in range(len(wordTensor)):
    print("a", end="") if wordTensor[i] == 1 else None
    print("b", end="") if wordTensor[i] == 2 else None
    print("c", end="") if wordTensor[i] == 3 else None
print()
# print(wordTensor.tolist())

for t in range(wordTokens.shape[1]):
    true_labels = wordTargets[0, t].int().tolist()
    pred_labels = preds[0, t].tolist()

    print(f"{t:02d}  True: {true_labels}   Pred: {pred_labels}   Correct:{true_labels == pred_labels}")

print(f"Seq Acc: {seq_acc.item():.4f}  (fraction of prefixes with an exactly-correct descent set)")

### Attention patterns per each head on ONE word
- prints 4 graphs

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)


# Move to test device, because it has more memory
chosenWord = train_tokens[wordIndex].unsqueeze(0).to(device1)  # shape (1, 22)
print(chosenWord)


# Generate labeled tokens with position

# ex: word len 22: str_tokens = [0,...,21]
str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

# Visualize
n_heads = model.cfg.n_heads
n_layers = model.cfg.n_layers
for layer in range(n_layers):
    for head in range(n_heads):
        # gets the attention pattern for layer 0, pick 1 word from the batch and look at 1 head for it
        # detatch tensor from computation graph (no gradients), move tensor to cpu to make imshow heatmap
        attn_single_head = cache["pattern", layer][wordIndex, head].detach().cpu()
        imshow(
            attn_single_head,
            x=str_tokens,
            y=str_tokens,
            xaxis="Key (Attended To)",
            yaxis="Query (Paying Attention)",
            title=f"Layer {layer} Head {head} Attention Pattern (Positional Tokens)",
            aspect=None
        )


### Attention Pattern for wordIndex over attention Heads

In [ ]:
# Attention pattern for a specific word (x: generators, y: attention heads)
# Assuming train_tokens is shape (batch, seq_len), and you're using batch 0?

token_strs = [str(x) for x in chosenWord.squeeze(0).tolist()]

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer][wordIndex][:, clsDex, :]
    imshow(
        attention,
        title=f"Attention Pattern (for word {wordIndex}) over Attention Heads in layer {layer}",
        xaxis="Source",
        yaxis="Head",
        #x=[str(i) for i in range(train_tokens.shape[1])],
        xlabels=token_strs,
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

print([str(i) for i in range(train_tokens.shape[1])])
print(token_strs)

## Positional Encoding
Rotary positional embeddings (`positional_embedding_type="rotary"`): position is **not** a learned vector added to the residual stream (there is no `model.pos_embed`). Instead, fixed sine/cosine rotations are applied to the query and key vectors inside every attention layer, so attention scores depend on the **relative** offset between positions. This is what lets the model learn one position-independent rule and apply it at every prefix (see REPORT.md, exp01).
- **Heatmaps** of the fixed rotary basis `rotary_sin` / `rotary_cos` (position x rotary dimension).
- **Cosine-similarity** matrix between per-position rotary vectors — banded/Toeplitz structure that depends only on the position *difference*.

## Misclassification Graphs
- Counting how many times the model predicts an incorrect descent set at each token index (position). With causal masking each index is an independent prediction over its prefix.

### Train Misclassification Graphs:

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt


def position_error_counts(model, tokens, targets, mask):
    """
    Per-token-index misclassification counts.

    With causal masking, the prediction at index i depends only on the prefix
    s_1..s_i, so "token index" i is a well-defined position. A position counts as
    incorrect when its predicted descent set differs from the target in any
    generator bit (same exact-set criterion as descent_accuracy_fn). Padding
    positions (mask == 0) are excluded.

    Returns (incorrect_per_pos, total_per_pos), each shape (seq_len,) on CPU.
    """
    logits = getLogits(model, tokens, getCache=False)        # [B, S, n_gen]
    preds  = (logits > 0).float()
    correct_bits = (preds == targets).float().sum(dim=-1)    # [B, S]
    exact  = (correct_bits == logits.size(-1)).float()       # [B, S] 1 = all bits right
    m = mask.float()                                         # [B, S]
    incorrect_per_pos = ((1.0 - exact) * m).sum(dim=0)       # [S]
    total_per_pos     = m.sum(dim=0)                         # [S]
    del logits
    return incorrect_per_pos.cpu(), total_per_pos.cpu()


# --- Per-sequence stats (still used by the sampling + "Other Graphs" cells) ---
words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

# Per-sequence descent accuracy for all train examples (1.0 = every prefix's descent set correct)
train_seq_acc = getPredictions(model, train_tokens, train_targets, train_mask)

for i in range(len(train_tokens)):
    input_seq = train_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(train_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if train_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

# --- Per-token-index error counts (the misclassification-by-position view) ---
train_incorrect, train_total = position_error_counts(model, train_tokens, train_targets, train_mask)
print("Train incorrect predictions per token index:")
print(train_incorrect.int().tolist())
print(f"Total incorrect positions: {int(train_incorrect.sum())} / {int(train_total.sum())} valid positions")

In [ ]:
# Per-sequence misclassified list (used by the attention-head visualization below)
misclassified = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Misclassified Train Sequences:")
for word, length, acc, idx in random.sample(misclassified, min(20, len(misclassified))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

positions = list(range(len(train_incorrect)))
counts    = train_incorrect.int().tolist()

# Plot 1: Raw count of incorrect predictions at each token index
plt.figure(figsize=(12, 6))
bars = plt.bar(positions, counts, edgecolor='black')
for bar, c in zip(bars, counts):
    if c > 0:
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 str(c), ha='center', va='bottom', fontsize=8)
plt.title("Incorrect Predictions by Token Index (Train)")
plt.xlabel("Token Index (position in sequence)")
plt.ylabel("Number of Incorrect Predictions")
plt.xticks(positions)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

# Plot 2: Misclassification rate at each token index (relative to valid positions there)
percent = [100 * i / t if t > 0 else 0.0
           for i, t in zip(train_incorrect.tolist(), train_total.tolist())]
plt.figure(figsize=(12, 6))
bars = plt.bar(positions, percent, edgecolor='black')
for bar, c in zip(bars, counts):
    if c > 0:
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 str(c), ha='center', va='bottom', fontsize=8)
plt.title("Misclassification Rate by Token Index (Train)")
plt.xlabel("Token Index (position in sequence)")
plt.ylabel("Incorrect Rate (%)")
plt.xticks(positions)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

### Test Misclassification Graphs

In [ ]:
words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

test_tokens   = test_tokens.to(device1)
test_seq_acc  = getPredictions(model, test_tokens, test_targets, test_mask)

for i in range(len(test_tokens)):
    input_seq = test_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(test_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if test_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

# Per-sequence misclassified list (used by the attention-head visualization below)
misclassified = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Misclassified Test Sequences:")
for word, length, acc, idx in random.sample(misclassified, min(20, len(misclassified))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

# Per-token-index error counts
test_incorrect, test_total = position_error_counts(model, test_tokens, test_targets, test_mask)
print("\nTest incorrect predictions per token index:")
print(test_incorrect.int().tolist())
print(f"Total incorrect positions: {int(test_incorrect.sum())} / {int(test_total.sum())} valid positions")

positions = list(range(len(test_incorrect)))
counts    = test_incorrect.int().tolist()

# Plot 1: Raw count of incorrect predictions at each token index
plt.figure(figsize=(12, 6))
bars = plt.bar(positions, counts, edgecolor='black')
for bar, c in zip(bars, counts):
    if c > 0:
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 str(c), ha='center', va='bottom', fontsize=8)
plt.title("Incorrect Predictions by Token Index (Test)")
plt.xlabel("Token Index (position in sequence)")
plt.ylabel("Number of Incorrect Predictions")
plt.xticks(positions)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

# Plot 2: Misclassification rate at each token index (relative to valid positions there)
percent = [100 * i / t if t > 0 else 0.0
           for i, t in zip(test_incorrect.tolist(), test_total.tolist())]
plt.figure(figsize=(12, 6))
bars = plt.bar(positions, percent, edgecolor='black')
for bar, c in zip(bars, counts):
    if c > 0:
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 str(c), ha='center', va='bottom', fontsize=8)
plt.title("Misclassification Rate by Token Index (Test)")
plt.xlabel("Token Index (position in sequence)")
plt.ylabel("Incorrect Rate (%)")
plt.xticks(positions)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

### Other Graphs
- check on misclassified
- attention heads of random selection of misclassified train inputs

In [ ]:
print(f"total word count: {sum(total_by_length.values())}")
print(f"total misclassified count: {sum(misclassified_by_length.values())}")

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)

# Make sure there are at least 4 items to sample from
sample_size = min(4, len(misclassified))

# Take random sample of misclassified items
sample = random.sample(misclassified, sample_size)

misclassified_index = [index for _, _, _, index in sample]
# NOTE: the big `cache` above was computed on train_tokens, but these indices point
# into the TEST set — so run the model on each chosen test word to get its own cache.
for mis_index in misclassified_index:
    chosenWord = test_tokens[mis_index].unsqueeze(0).to(device1)  # shape (1, 22)
    _, word_cache = getLogits(model, chosenWord, getCache=True)

    # Generate labeled tokens with position

    # ex: word len 22: str_tokens = [0,...,21]
    str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

    # Visualize
    n_heads = model.cfg.n_heads
    n_layers = model.cfg.n_layers
    for layer in range(n_layers):
        for head in range(n_heads):
            # this word is batch entry 0 of its own single-word cache
            attn_single_head = word_cache["pattern", layer][0, head].detach().cpu()

            imshow(
                attn_single_head,
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} Attention Pattern (for test word {mis_index})(Positional Tokens)"
            )

In [ ]:
# Rotary positional encoding: fixed (not learned) sin/cos rotation angles applied to
# queries/keys in every attention layer. rotary_sin/rotary_cos: [n_ctx, d_head].
# Low dimensions rotate fast (fine-grained local position), high dimensions slowly.
rot_sin = model.blocks[0].attn.rotary_sin.detach().cpu()   # [n_ctx, d_head]
rot_cos = model.blocks[0].attn.rotary_cos.detach().cpu()   # [n_ctx, d_head]
print("rotary_sin", tuple(rot_sin.shape), "| rotary_cos", tuple(rot_cos.shape))

imshow(
    rot_sin,
    xaxis="Rotary Dimension",
    yaxis="Position",
    ylabels=[str(i) for i in range(rot_sin.shape[0])],
    title="Rotary Basis: sin (Position x rotary dim)",
    aspect=None,
)
imshow(
    rot_cos,
    xaxis="Rotary Dimension",
    yaxis="Position",
    ylabels=[str(i) for i in range(rot_cos.shape[0])],
    title="Rotary Basis: cos (Position x rotary dim)",
    aspect=None,
)

In [ ]:
# Cosine similarity between per-position rotary vectors concat(cos_p, sin_p).
# Because the rotary basis is a bank of fixed-frequency rotations, similarity depends
# only on the position DIFFERENCE p - q: the matrix is banded/Toeplitz. This built-in
# relative structure is exactly what removed the length cliff vs learned-absolute
# embeddings (REPORT.md, exp01) — no structure here is learned.
rot = torch.cat([rot_cos, rot_sin], dim=-1)            # [n_ctx, 2*d_head]
rot_norm = rot / rot.norm(dim=-1, keepdim=True)
pos_cos_sim = rot_norm @ rot_norm.T                    # [n_ctx, n_ctx]

pos_labels = [str(i) for i in range(rot.shape[0])]
imshow(
    pos_cos_sim,
    x=pos_labels,
    y=pos_labels,
    xaxis="Position",
    yaxis="Position",
    title="Cosine Similarity Between Rotary Position Vectors (fixed, relative)",
    aspect=None,
)